# Mi perfil de puntos de trazado GTFS

Consulté ambos snapshots mediante Genie el 11 de noviembre de 2026. Estas nueve consultas definitivas son de lectura; sus 55 filas estructuradas se recuperaron completas. El [informe](../../../docs/shape_points_profile_2025_2026.md) interpreta los resultados y la [contrato implementado](../../../docs/contracts/shape_points.md) define las reglas revisadas.

Una fila de shapes es un punto numerado de un trazado, no una parada ni una posición observada. Conservo SQL verificable sin outputs en Git. La ejecución de cada consulta mediante Genie no equivale a haber ejecutado este cuaderno entero en Databricks. Este notebook no se añade al Job ni publica Silver.

Los nombres workspace.gtfs_bogota, los snapshots y el Volume son los valores actuales del proyecto; deberán configurarse al reutilizarlo en otro entorno. Filtrar por hash y comprobar versión/ruta evita ocultar procedencia incoherente. Las consultas no fijan una versión Delta ni constituyen una transacción conjunta de las tablas.


## Bloque 1 — Esquema disponible

¿Qué columnas y tipos tienen las fuentes? Reviso 23 campos entre shapes, trips y auditoría. Los cuatro campos de negocio de shapes son string en Bronze.


In [ ]:
%sql
-- BLOQUE 1 — Perfil de solo lectura.
select table_name, column_name, data_type, ordinal_position
from workspace.information_schema.columns
where table_schema = 'gtfs_bogota'
  and table_name in ('bronze_gtfs_shapes','bronze_gtfs_trips','audit_gtfs_source_schema_observations')
order by table_name, ordinal_position;


## Bloque 2 — Campos realmente publicados

¿Qué encabezado vino en cada versión? Hay cuatro observaciones de shapes/trips; shape_dist_traveled no aparece en ninguno de los dos archivos shapes.txt.


In [ ]:
%sql
-- BLOQUE 2 — Perfil de solo lectura.
select source_feed_version, source_sha256, source_file_name, source_columns_json
from workspace.gtfs_bogota.audit_gtfs_source_schema_observations
where source_file_name in ('shapes.txt','trips.txt')
and ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
order by source_feed_version, source_file_name;


## Bloque 3 — Completitud por campo

¿Hay nulos, vacíos o espacios? Las ocho filas cubren cuatro campos por snapshot. Nulo, vacío y solo espacios son categorías separadas; los espacios extremos son un diagnóstico que puede solaparse.


In [ ]:
%sql
-- BLOQUE 3 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
metrics as (
select source_feed_version, source_sha256, count(*) as snapshot_rows,
  count_if(shape_id is null) as n0,
  count_if(shape_id='') as e0,
  count_if(shape_id<>'' and shape_id rlike r'(?U)^\s*$') as w0,
  count_if(shape_id rlike r'(?U)^\s|\s$') as t0,
  count(distinct case when shape_id is not null and not (shape_id rlike r'(?U)^\s*$') then shape_id end) as d0,
  count_if(shape_pt_sequence is null) as n1,
  count_if(shape_pt_sequence='') as e1,
  count_if(shape_pt_sequence<>'' and shape_pt_sequence rlike r'(?U)^\s*$') as w1,
  count_if(shape_pt_sequence rlike r'(?U)^\s|\s$') as t1,
  count(distinct case when shape_pt_sequence is not null and not (shape_pt_sequence rlike r'(?U)^\s*$') then shape_pt_sequence end) as d1,
  count_if(shape_pt_lat is null) as n2,
  count_if(shape_pt_lat='') as e2,
  count_if(shape_pt_lat<>'' and shape_pt_lat rlike r'(?U)^\s*$') as w2,
  count_if(shape_pt_lat rlike r'(?U)^\s|\s$') as t2,
  count(distinct case when shape_pt_lat is not null and not (shape_pt_lat rlike r'(?U)^\s*$') then shape_pt_lat end) as d2,
  count_if(shape_pt_lon is null) as n3,
  count_if(shape_pt_lon='') as e3,
  count_if(shape_pt_lon<>'' and shape_pt_lon rlike r'(?U)^\s*$') as w3,
  count_if(shape_pt_lon rlike r'(?U)^\s|\s$') as t3,
  count(distinct case when shape_pt_lon is not null and not (shape_pt_lon rlike r'(?U)^\s*$') then shape_pt_lon end) as d3
from shapes group by source_feed_version,source_sha256
)
select source_feed_version,source_sha256,snapshot_rows,
inline(array(named_struct('field_name','shape_id','null_count',n0,'empty_count',e0,'whitespace_only_count',w0,'edge_whitespace_count',t0,'distinct_non_blank_count',d0),
named_struct('field_name','shape_pt_sequence','null_count',n1,'empty_count',e1,'whitespace_only_count',w1,'edge_whitespace_count',t1,'distinct_non_blank_count',d1),
named_struct('field_name','shape_pt_lat','null_count',n2,'empty_count',e2,'whitespace_only_count',w2,'edge_whitespace_count',t2,'distinct_non_blank_count',d2),
named_struct('field_name','shape_pt_lon','null_count',n3,'empty_count',e3,'whitespace_only_count',w3,'edge_whitespace_count',t3,'distinct_non_blank_count',d3)))
from metrics order by source_feed_version,field_name;


## Bloque 4 — Tipos, rangos y procedencia

¿Puedo tipar los valores y confiar en el snapshot? Cuento filas y trazados, verifico versión, ruta y timestamp, convierto con try_cast y reviso ID, secuencia y rangos geográficos. No verifico el polígono de Bogotá.


In [ ]:
%sql
-- BLOQUE 4 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
)
select source_feed_version,source_sha256,count(*) as input_rows,
count(distinct shape_id) as distinct_shapes,
count_if(actual_feed_version is null or actual_feed_version<>source_feed_version) as wrong_version_rows,
count_if(source_file_path is null or source_file_path<>concat('/Volumes/workspace/gtfs_bogota/staging_gtfs/gtfs_static/feed_version=',source_feed_version,'/sha256=',source_sha256,'/shapes.txt')) as wrong_path_rows,
count_if(bronze_loaded_at_utc is null) as missing_load_timestamp_rows,
count_if(shape_id is null or shape_id rlike r'(?U)^\s*$|^\s|\s$|[\p{Cc}]') as invalid_id_rows,
count_if(shape_pt_sequence is null or not (shape_pt_sequence rlike r'^[0-9]+$') or seq is null or seq<0) as invalid_sequence_rows,
count_if(lat is null or isnan(lat) or abs(lat)>90) as invalid_latitude_rows,
count_if(lon is null or isnan(lon) or abs(lon)>180) as invalid_longitude_rows,
count_if(lat=0 and lon=0) as zero_zero_rows,
min(seq) as min_sequence,max(seq) as max_sequence,
min(lat) as min_latitude,max(lat) as max_latitude,
min(lon) as min_longitude,max(lon) as max_longitude
from shapes group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 5 — Copias, conflictos y colisiones

¿Se repite una clave? Distingo copias de negocio idénticas, atributos distintos para una clave y secuencias que colisionan al convertirlas a int. Excluyo columnas técnicas de la igualdad de negocio.


In [ ]:
%sql
-- BLOQUE 5 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
raw_groups as (
select source_feed_version,source_sha256,shape_id,shape_pt_sequence,shape_pt_lat,shape_pt_lon,count(*) as copies
from shapes
group by source_feed_version,source_sha256,shape_id,shape_pt_sequence,shape_pt_lat,shape_pt_lon
),
raw_keys as (
select source_feed_version,source_sha256,shape_id,shape_pt_sequence,sum(copies) as source_rows,count(*) as business_variants
from raw_groups group by source_feed_version,source_sha256,shape_id,shape_pt_sequence
),
typed_keys as (
select source_feed_version,source_sha256,shape_id,seq,count(distinct shape_pt_sequence) as raw_sequence_variants
from shapes where seq is not null and seq>=0
group by source_feed_version,source_sha256,shape_id,seq
),
copies as (
select source_feed_version,source_sha256,count_if(copies>1) as identical_copy_groups,sum(copies-1) as identical_copy_excess
from raw_groups group by source_feed_version,source_sha256
),
conflicts as (
select source_feed_version,source_sha256,count_if(source_rows>1) as repeated_raw_keys,count_if(business_variants>1) as conflicting_raw_keys
from raw_keys group by source_feed_version,source_sha256
),
casts as (
select source_feed_version,source_sha256,count_if(raw_sequence_variants>1) as cast_key_collision_groups
from typed_keys group by source_feed_version,source_sha256
)
select c.source_feed_version,c.source_sha256,c.identical_copy_groups,c.identical_copy_excess,
k.repeated_raw_keys,k.conflicting_raw_keys,t.cast_key_collision_groups
from copies c join conflicts k using(source_feed_version,source_sha256)
join casts t using(source_feed_version,source_sha256)
order by c.source_feed_version;


## Bloque 6 — Estructura de cada trazado

¿Cuántos puntos y coordenadas distintas dibujan cada forma? Todos tienen al menos cuatro coordenadas distintas. Las secuencias pueden empezar en valores diferentes. Extendí esta consulta para revisar degeneración; conservo aquí solo su versión definitiva.


In [ ]:
%sql
-- BLOQUE 6 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
per_shape as (
select source_feed_version,source_sha256,shape_id,count(*) as points,
count(distinct seq) as sequence_points,count(distinct struct(lat,lon)) as coordinate_points,
min(seq) as first_sequence,max(seq) as last_sequence
from shapes group by source_feed_version,source_sha256,shape_id
)
select source_feed_version,source_sha256,count(*) as shapes,
sum(points) as points,min(points) as min_points,max(points) as max_points,
min(coordinate_points) as min_unique_coordinates,
count_if(coordinate_points<2) as shapes_with_fewer_than_two_unique_coordinates,
percentile_approx(points,array(0.5,0.95),10000) as p50_p95_points,
count_if(points<2) as shapes_with_fewer_than_two_points,
count_if(first_sequence=0) as shapes_starting_at_zero,
count_if(first_sequence=1) as shapes_starting_at_one,
count_if(first_sequence not in (0,1)) as shapes_starting_elsewhere,
count_if(points>coordinate_points) as shapes_with_repeated_coordinates,
sum(points-coordinate_points) as repeated_coordinate_excess
from per_shape group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 7 — Segmentos entre puntos

¿Qué ocurre entre un punto y el siguiente? Ordeno por secuencia dentro del snapshot y trazado. Reviso huecos y coordenadas consecutivas iguales. La distancia recta calculada usa una esfera de radio 6.371.000 m y es diagnóstica; no es distancia publicada, longitud vial real ni velocidad. Solo se analizan secuencias/coordenadas válidas; el bloque 4 confirmó cero inválidos en estos snapshots.


In [ ]:
%sql
-- BLOQUE 7 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
ordered as (
select source_feed_version,source_sha256,shape_id,seq,lat,lon,
lag(seq) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_seq,
lag(lat) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_lat,
lag(lon) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_lon
from shapes
where seq>=0 and lat between -90 and 90 and lon between -180 and 180
),
distances as (
select source_feed_version,source_sha256,seq,previous_seq,lat,lon,previous_lat,previous_lon,
2*6371000*asin(sqrt(least(1.0,greatest(0.0,
pow(sin(radians(lat-previous_lat)/2),2)
+cos(radians(previous_lat))*cos(radians(lat))*pow(sin(radians(lon-previous_lon)/2),2)
)))) as straight_line_m
from ordered where previous_seq is not null
)
select source_feed_version,source_sha256,count(*) as segments,
count_if(seq>previous_seq+1) as sequence_gaps,
count_if(seq<=previous_seq) as non_increasing_sequences,
count_if(lat=previous_lat and lon=previous_lon) as zero_length_segments,
round(max(straight_line_m),3) as max_straight_line_m,
percentile_approx(straight_line_m,array(0.5,0.95,0.99),10000) as p50_p95_p99_straight_line_m
from distances group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 8 — Referencias desde trips

¿Los viajes apuntan a trazados existentes? Uno contra IDs de trazado distintos para evitar multiplicar viajes. Hay cuatro trazados sin viajes referenciados en 2025; no los elimino. Esta consulta no valida trips íntegramente.


In [ ]:
%sql
-- BLOQUE 8 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
trips as (
select s.feed_version as source_feed_version,b.source_sha256,b.shape_id
from workspace.gtfs_bogota.bronze_gtfs_trips b
join snapshots s on b.source_sha256=s.sha256 and b.source_feed_version=s.feed_version
),
shape_ids as (
select distinct source_feed_version,source_sha256,shape_id from shapes
),
trip_metrics as (
select t.source_feed_version,t.source_sha256,count(*) as trip_rows,
count_if(t.shape_id is null or t.shape_id rlike r'(?U)^\s*$') as trips_without_shape_id,
count_if(t.shape_id is not null and not(t.shape_id rlike r'(?U)^\s*$') and p.shape_id is null) as trips_with_unknown_shape_id,
count(distinct t.shape_id) as referenced_shape_ids,
count(distinct case when t.shape_id is not null and not(t.shape_id rlike r'(?U)^\s*$') and p.shape_id is null then t.shape_id end) as unknown_shape_ids
from trips t left join shape_ids p using(source_feed_version,source_sha256,shape_id)
group by t.source_feed_version,t.source_sha256
),
trip_ids as (
select distinct source_feed_version,source_sha256,shape_id from trips
),
unused as (
select p.source_feed_version,p.source_sha256,count(*) as shapes_without_trip_reference
from shape_ids p left anti join trip_ids t using(source_feed_version,source_sha256,shape_id)
group by p.source_feed_version,p.source_sha256
)
select t.source_feed_version,t.source_sha256,t.trip_rows,t.trips_without_shape_id,
t.trips_with_unknown_shape_id,t.referenced_shape_ids,t.unknown_shape_ids,
coalesce(u.shapes_without_trip_reference,0) as shapes_without_trip_reference
from trip_metrics t left join unused u using(source_feed_version,source_sha256)
order by t.source_feed_version;


## Bloque 9 — Contexto de los segmentos largos

¿Qué modo de transporte usa los cinco segmentos más largos de cada snapshot? Relaciono vía trips con routes Silver dentro de versión/hash y recupero diez filas. El máximo pertenece a C_1 con route_type 6; no impongo un umbral de rechazo común a cable y bus.


In [ ]:
%sql
-- BLOQUE 9 — Perfil de solo lectura.
with snapshots as (
  select * from values
    ('2025-10-28','07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952'),
    ('2026-09-29','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
    as s(feed_version,sha256)
),
shapes as (
  select s.feed_version as source_feed_version, b.source_sha256,
    b.source_feed_version as actual_feed_version,
    b.shape_id, b.shape_pt_sequence, b.shape_pt_lat, b.shape_pt_lon,
    b.source_file_path, b.bronze_loaded_at_utc,
    try_cast(b.shape_pt_sequence as int) as seq,
    try_cast(b.shape_pt_lat as double) as lat,
    try_cast(b.shape_pt_lon as double) as lon
  from workspace.gtfs_bogota.bronze_gtfs_shapes b
  join snapshots s on b.source_sha256=s.sha256
),
ordered as (
select source_feed_version,source_sha256,shape_id,seq,lat,lon,
lag(seq) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_seq,
lag(lat) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_lat,
lag(lon) over(partition by source_feed_version,source_sha256,shape_id order by seq) as previous_lon
from shapes
where seq>=0 and lat between -90 and 90 and lon between -180 and 180
),
distances as (
select source_feed_version,source_sha256,shape_id,seq,previous_seq,lat,lon,previous_lat,previous_lon,
2*6371000*asin(sqrt(least(1.0,greatest(0.0,
pow(sin(radians(lat-previous_lat)/2),2)
+cos(radians(previous_lat))*cos(radians(lat))*pow(sin(radians(lon-previous_lon)/2),2)
)))) as straight_line_m
from ordered where previous_seq is not null
),
ranked as (
select source_feed_version,source_sha256,shape_id,seq,previous_seq,straight_line_m,
row_number() over(partition by source_feed_version,source_sha256 order by straight_line_m desc,shape_id,seq) as rank_in_snapshot
from distances
),
route_types as (
select t.source_feed_version,t.source_sha256,t.shape_id,
array_sort(collect_set(r.route_type)) as route_types
from workspace.gtfs_bogota.bronze_gtfs_trips t
join snapshots s on t.source_feed_version=s.feed_version and t.source_sha256=s.sha256
left join workspace.gtfs_bogota.silver_gtfs_routes r
on t.source_feed_version=r.source_feed_version and t.source_sha256=r.source_sha256 and t.route_id=r.route_id
group by t.source_feed_version,t.source_sha256,t.shape_id
)
select p.source_feed_version,p.source_sha256,p.shape_id,p.previous_seq,p.seq,
round(p.straight_line_m,3) as straight_line_m,r.route_types
from ranked p left join route_types r using(source_feed_version,source_sha256,shape_id)
where p.rank_in_snapshot<=5
order by p.source_feed_version,p.rank_in_snapshot;


## Cómo continúa

Ya revisé las reglas e implementé shape_points_v1 localmente. Los puntos con coordenadas repetidas y los trazados sin referencia se conservan; las reglas de advertencia quedarán en el contrato. El mapeo de shapes.txt a shape_points será explícito. La transformación reutiliza notebook 10. La plantilla añade su tarea. Las 36 pruebas seleccionadas pasaron sin omisiones. La primera publicación sigue pendiente.
